# Week 15: Graph Neural Networks & Geometric Deep Learning
## Notebook 06: Differentiable Graph Pooling (DiffPool) & Whole-Graph Classification from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In computer vision, CNNs use MaxPooling layers ($2 \times 2$) to hierarchically downsample pixel grids into coarser abstract features.
However, in graphs:
- Graphs have irregular topologies with no natural coordinate grid.
- Naive global pooling (averaging all node embeddings) discards complex hierarchical sub-structures (e.g. functional chemical groups in a drug molecule).

In 2018, Ying et al. (Stanford) introduced **DiffPool (Differentiable Pooling)**:
- Learns a **Soft Cluster Assignment Matrix ($S$)** using a parallel GNN.
- Coarsens a graph of $N$ nodes into $K$ community super-nodes ($K < N$).
- Enables deep hierarchical GNN architectures for **whole-graph classification** (e.g. predicting molecular toxicity or social community networks).

---

### 2. Mathematical Foundations: DiffPool Coarsening
At layer $l$ with adjacency matrix $A^{(l)} \in \mathbb{R}^{N_l \times N_l}$ and node features $X^{(l)} \in \mathbb{R}^{N_l \times D_l}$:

1. **Embedding GNN:** $Z^{(l)} = \text{GNN}_{l, \text{embed}}(A^{(l)}, X^{(l)}) \in \mathbb{R}^{N_l \times D_{\text{out}}}$
2. **Pooling Assignment GNN:**
   $$S^{(l)} = \text{Softmax}\left( \text{GNN}_{l, \text{pool}}(A^{(l)}, X^{(l)}) \right) \in \mathbb{R}^{N_l \times N_{l+1}}$$
   Where $S_{ij}^{(l)}$ represents the probability that node $i$ belongs to cluster $j$ in the coarser layer!

3. **Coarsened Node Features:**
   $$X^{(l+1)} = (S^{(l)})^T Z^{(l)} \in \mathbb{R}^{N_{l+1} \times D_{\text{out}}}$$
4. **Coarsened Adjacency Matrix:**
   $$A^{(l+1)} = (S^{(l)})^T A^{(l)} S^{(l)} \in \mathbb{R}^{N_{l+1} \times N_{l+1}}$$

In [ ]:
import numpy as np

np.random.seed(42)

# Step 1: Define a 6-node Molecular Graph
# 2 Distinct Functional Groups (Nodes 0,1,2 in Group A; Nodes 3,4,5 in Group B)
N = 6
D_in = 4   # Node feature dimension
K_clusters = 2 # Coarsen from 6 nodes down to 2 super-nodes!

A = np.array([
    [1, 1, 1, 0, 0, 0],
    [1, 1, 1, 0, 0, 0],
    [1, 1, 1, 1, 0, 0], # Bridge edge between 2 and 3
    [0, 0, 1, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1]
], dtype=float)

X = np.random.randn(N, D_in)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

# Step 2: Implement DiffPool Layer
# A. Feature Embeddings Z = A * X * W_embed
W_embed = np.random.randn(D_in, 8) * 0.4
Z = A @ X @ W_embed # (6, 8)

# B. Soft Cluster Assignment Matrix S = Softmax(A * X * W_pool)
W_pool = np.random.randn(D_in, K_clusters) * 0.4
S_raw = A @ X @ W_pool
S = softmax(S_raw) # (6, 2)

print("--- Soft Cluster Assignment Matrix S (6 nodes -> 2 clusters) ---")
print(np.round(S, 3))

# Step 3: Compute Coarsened Representations
# Coarsened Node Features: X_next = S.T @ Z
X_coarsened = S.T @ Z  # (2, 8)

# Coarsened Adjacency: A_next = S.T @ A @ S
A_coarsened = S.T @ A @ S # (2, 2)

print(f"\nOriginal Graph Shape:    Adjacency {A.shape}, Features {X.shape}")
print(f"Coarsened Graph Shape:   Adjacency {A_coarsened.shape}, Features {X_coarsened.shape}")
print("\nCoarsened Super-Node Adjacency Matrix:")
print(np.round(A_coarsened, 3))

### 3. Step-by-Step Code Walkthrough
1. **Parallel GNN Pathways:** One GNN computes node feature representations $Z$, while a secondary GNN predicts soft community cluster assignments $S$.
2. **Matrix Coarsening:** $S^T A S$ condenses the $6 \times 6$ adjacency matrix into a $2 \times 2$ super-node graph, preserving inter-community connectivity.

---

### 4. Industry Applications & Trade-offs
- **Biochemical Drug Screening:** Classifying whether complex protein-ligand graphs exhibit toxic biological activity.
- **Trade-off:** Computing dense matrix products $S^T A S$ consumes $O(K N^2)$ memory, which requires sparse approximations for very large graphs.

---

### 5. Top Interview Questions & Answers
**Q1: Why is an auxiliary Link Prediction loss used during DiffPool training?**
> **Answer:** Without auxiliary constraints, the pooling GNN can collapse all nodes into a single cluster. DiffPool uses an auxiliary link prediction loss ($L_{LP} = \|A - S S^T\|_F$) and entropy regularization to ensure nodes connected by edges are mapped to the same super-node cluster.

**Q2: What is the difference between Node-Level and Graph-Level tasks in GNNs?**
> **Answer:** Node-level tasks predict properties of individual nodes (e.g. user fraud). Graph-level tasks require a single classification or regression score for the entire graph (e.g. molecular solubility), requiring global pooling or hierarchical coarsening (DiffPool).